# MDG-LiteNet — Full Working Implementation
### Explainable Magnification-Aware Deformable Graph Learning with Quantum-Aware Super Eagle Optimization

This notebook implements a practical, end-to-end version of the uploaded workflow:
**Macenko-style stain normalization → Otsu tissue mask → multiscale patch seeds → MORPH-G graph → bounded DeformGraph → MAG-Fuse → QSEO-HPO → sparse channel head → knowledge distillation → 8-bit post-training quantization → Integrated Gradients / sampled Shapley / counterfactual explanations → patient-level evaluation**.

The workflow specifies BreakHis as the primary dataset, with patient-level separation, four magnifications, 224×336 input resolution, 28/56-pixel patches, maximum 120 graph nodes, 2 DeformGraph layers, MAG-Fuse, and a 20,000-parameter deployment budget. fileciteturn0file0L187-L200 fileciteturn0file0L396-L403

**Important:** the notebook does not fabricate paper-reported experimental results. Metrics printed by this notebook are computed from the dataset and training run you provide.


In [ ]:
# Cell 1 — Install dependencies
# Run once in Google Colab/Jupyter.

!pip -q install scikit-image opencv-python-headless pandas numpy matplotlib seaborn scikit-learn pillow tqdm networkx
!pip -q install torch torchvision

import sys, subprocess, pkgutil, os, random, math, json, time, warnings
warnings.filterwarnings("ignore")

print("Python:", sys.version)


In [ ]:
# Cell 2 — Imports and reproducibility

import os, re, glob, math, random, time, copy, json, hashlib
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, matthews_corrcoef, confusion_matrix,
    classification_report
)
from sklearn.model_selection import StratifiedGroupKFold
from skimage.color import rgb2lab, rgb2hsv
from skimage.filters import threshold_otsu
from skimage.segmentation import slic
from skimage.measure import regionprops, label

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## 1. Configuration

The source workflow uses 224×336 images, 28/56 patch widths, 0.5 tissue fraction, SLIC with five iterations, 12-dimensional morphological descriptors, maximum 120 nodes, two graph layers, latent dimension 8, temperature 0.5, candidate cap 24, and offset penalty 0.01. fileciteturn0file0L396-L403 fileciteturn0file0L449-L456


In [ ]:
# Cell 3 — User configuration

CONFIG = {
    # Set this to your extracted BreakHis root.
    # Expected: .../SOB/... image files, or any recursive folder structure.
    "DATA_ROOT": "/content/BreakHis",

    # Optional second/third external datasets.
    "BACH_ROOT": "",
    "IDC_ROOT": "",

    "OUTPUT_ROOT": "/content/MDG_LiteNet_Outputs",

    "IMAGE_SIZE": (224, 336),       # H, W
    "PATCH_SMALL": 28,
    "PATCH_LARGE": 56,
    "TISSUE_FRACTION": 0.50,
    "SLIC_ITERATIONS": 5,
    "SLIC_COMPACTNESS": 10,
    "MAX_NODES": 120,
    "K_NEIGHBORS": 3,

    "GRAPH_LAYERS": 2,
    "LATENT_DIM": 8,
    "TEMPERATURE": 0.5,
    "CANDIDATE_CAP": 24,
    "DEFORMED_NEIGHBORS": 3,
    "OFFSET_BOUND": 0.5,
    "OFFSET_PENALTY": 0.01,

    "MAG_DROP": 0.25,
    "FUSION_BALANCE": 0.05,
    "KEY_DIM": 16,

    "WIDTH": 32,
    "RANK": 8,
    "KEPT_CHANNELS": 16,
    "DROPOUT": 0.10,
    "DECISION_THRESHOLD": 0.50,

    "BATCH_SIZE": 8,
    "EPOCHS": 15,
    "PATIENCE": 5,
    "LR": 1e-3,
    "WEIGHT_DECAY": 1e-4,

    "NUM_WORKERS": 2,
    "MAX_IMAGES_PER_CLASS": None,

    # QSEO-HPO practical settings. Increase after confirming the notebook works.
    "QSEO_POPULATION": 6,
    "QSEO_ITERATIONS": 5,
    "QSEO_PROXY_EPOCHS": 2,

    "USE_MACENKO": True,
    "CACHE_GRAPHS": True
}

os.makedirs(CONFIG["OUTPUT_ROOT"], exist_ok=True)
print(json.dumps(CONFIG, indent=2))


## 2. Dataset discovery

The paper describes BreakHis as the primary dataset with 7,909 images from 82 patients and four magnifications, mapped to benign=0 and malignant=1. Patient-disjoint partitioning is required. fileciteturn0file0L187-L200

This cell is deliberately flexible: it searches image files recursively and infers labels/patient IDs/magnification from common BreakHis naming conventions. If your directory uses a different naming scheme, edit `parse_breakhis_path()`.


In [ ]:
# Cell 4 — Dataset indexing

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def parse_breakhis_path(path):
    p = str(path).replace("\\", "/")
    low = p.lower()

    # Label
    if "/benign/" in low or "/b/" in low or "_b_" in low:
        label = 0
    elif "/malignant/" in low or "/m/" in low or "_m_" in low:
        label = 1
    else:
        label = 1 if re.search(r"(^|[_/\\-])m(alignant)?([_/\\-]|$)", low) else (
            0 if re.search(r"(^|[_/\\-])b(enign)?([_/\\-]|$)", low) else None
        )

    # Magnification
    mag = None
    for m in [40, 100, 200, 400]:
        if re.search(rf"(?<!\d){m}x?(?!\d)", low):
            mag = m
            break

    # Patient ID — common BreakHis pattern: SOB_<benign/malignant>_<organ>_<id>...
    patient = None
    m = re.search(r"(SOB_[^/\\]+?_\d+)", p, re.I)
    if m:
        patient = m.group(1)
    else:
        # fallback: parent folder containing SOB
        parts = p.split("/")
        candidates = [x for x in parts if x.upper().startswith("SOB")]
        patient = candidates[-1] if candidates else Path(path).parent.name

    return label, mag, patient

def build_index(root, max_per_class=None):
    root = Path(root)
    files = [p for p in root.rglob("*") if p.suffix.lower() in IMAGE_EXTS]
    rows = []
    counts = Counter()
    for p in files:
        y, mag, patient = parse_breakhis_path(p)
        if y is None:
            continue
        if max_per_class is not None and counts[y] >= max_per_class:
            continue
        rows.append({
            "path": str(p),
            "label": int(y),
            "patient": str(patient),
            "magnification": mag if mag is not None else -1
        })
        counts[y] += 1

    df = pd.DataFrame(rows)
    if len(df):
        df = df.drop_duplicates("path").reset_index(drop=True)
    return df

df = build_index(CONFIG["DATA_ROOT"], CONFIG["MAX_IMAGES_PER_CLASS"])
print("Images:", len(df))
if len(df):
    print(df.head())
    print("\nLabels:", df["label"].value_counts().to_dict())
    print("Patients:", df["patient"].nunique())
    print("Magnifications:", df["magnification"].value_counts().to_dict())
else:
    print("No images found. Set CONFIG['DATA_ROOT'] to your extracted dataset.")


In [ ]:
# Cell 5 — Optional dataset sanity check

if len(df):
    fig, ax = plt.subplots(figsize=(7,4))
    df["magnification"].value_counts().sort_index().plot(kind="bar", ax=ax)
    ax.set_title("Images by magnification")
    ax.set_xlabel("Magnification")
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()

    print(classification_report(
        df["label"], df["label"], target_names=["Benign","Malignant"], zero_division=0
    ))


## 3. Macenko-style stain normalization

The workflow specifies optical-density conversion, a 0.15 OD threshold, SVD-based stain basis estimation, percentile-based stain concentrations, and reference statistics from a training image. fileciteturn0file0L323-L342

The implementation below is a self-contained Macenko-style normalizer with safe fallbacks for low-information images.


In [ ]:
# Cell 6 — Macenko stain normalization

def rgb_to_od(img):
    img = np.asarray(img).astype(np.float32)
    img = np.clip(img, 1, 255)
    return -np.log(img / 255.0)

def normalize_rows(x, eps=1e-8):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + eps)

def macenko_stain_matrix(img, od_threshold=0.15):
    OD = rgb_to_od(img).reshape(-1, 3)
    OD = OD[np.all(OD > od_threshold, axis=1)]
    if len(OD) < 100:
        return np.array([[0.65, 0.70, 0.29],
                         [0.07, 0.99, 0.11]], dtype=np.float32)

    cov = np.cov(OD, rowvar=False)
    vals, vecs = np.linalg.eigh(cov)
    order = np.argsort(vals)[::-1]
    vecs = vecs[:, order[:2]]

    proj = OD @ vecs
    angles = np.arctan2(proj[:,1], proj[:,0])
    min_ang, max_ang = np.percentile(angles, [1, 99])

    v1 = vecs @ np.array([np.cos(min_ang), np.sin(min_ang)])
    v2 = vecs @ np.array([np.cos(max_ang), np.sin(max_ang)])
    M = np.stack([v1, v2], axis=0)

    # Consistent H&E orientation
    M = normalize_rows(M)
    if M[0, 0] < M[1, 0]:
        M = M[::-1]

    return M.astype(np.float32)

def macenko_normalize(img, reference_matrix=None, od_threshold=0.15):
    img = np.asarray(img).astype(np.uint8)
    M = macenko_stain_matrix(img, od_threshold)

    if reference_matrix is None:
        reference_matrix = np.array([[0.65, 0.70, 0.29],
                                     [0.07, 0.99, 0.11]], dtype=np.float32)

    OD = rgb_to_od(img).reshape(-1, 3)
    C = np.linalg.pinv(M.T) @ OD.T

    # Robust concentration scaling
    max_c = np.percentile(C, 99, axis=1, keepdims=True)
    max_ref = np.array([1.0, 1.0], dtype=np.float32).reshape(2,1)
    C = C * (max_ref / np.maximum(max_c, 1e-6))

    OD_norm = reference_matrix.T @ C
    out = np.exp(-OD_norm).T.reshape(img.shape) * 255.0
    return np.clip(out, 0, 255).astype(np.uint8), M, C.T.reshape(img.shape[0], img.shape[1], 2)

def resize_image(img):
    return cv2.resize(img, (CONFIG["IMAGE_SIZE"][1], CONFIG["IMAGE_SIZE"][0]), interpolation=cv2.INTER_AREA)

def load_and_normalize(path):
    img = np.array(Image.open(path).convert("RGB"))
    img = resize_image(img)
    if CONFIG["USE_MACENKO"]:
        try:
            img, _, conc = macenko_normalize(img)
        except Exception:
            conc = np.zeros((*img.shape[:2],2), np.float32)
    else:
        conc = np.zeros((*img.shape[:2],2), np.float32)
    return img, conc


In [ ]:
# Cell 7 — Visualize stain normalization

if len(df):
    sample_path = df.iloc[0]["path"]
    raw = resize_image(np.array(Image.open(sample_path).convert("RGB")))
    norm, _, _ = macenko_normalize(raw)

    fig, axes = plt.subplots(1,2, figsize=(12,5))
    axes[0].imshow(raw); axes[0].set_title("Input")
    axes[1].imshow(norm); axes[1].set_title("Macenko-normalized")
    for ax in axes: ax.axis("off")
    plt.tight_layout()
    plt.show()


## 4. Otsu tissue masking and multiscale patch seeding

The source workflow uses HSV saturation and Otsu thresholding. Patches are retained when at least 50% of pixels are tissue; 28-pixel and 56-pixel patch scales provide cellular and architectural seeds. fileciteturn0file0L343-L356


In [ ]:
# Cell 8 — Tissue mask and patch extraction

def tissue_mask_otsu(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    sat = hsv[..., 1].astype(np.float32) / 255.0
    try:
        t = threshold_otsu(sat)
    except Exception:
        t = 0.10
    mask = (sat >= t).astype(np.uint8)
    # Small cleanup
    kernel = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask, float(t)

def patch_centers(mask, patch_w, fraction=0.5):
    H, W = mask.shape
    centers = []
    for y in range(0, H-patch_w+1, patch_w):
        for x in range(0, W-patch_w+1, patch_w):
            patch = mask[y:y+patch_w, x:x+patch_w]
            if patch.mean() >= fraction:
                centers.append((x + patch_w//2, y + patch_w//2))
    return centers

def get_multiscale_seeds(img):
    mask, threshold = tissue_mask_otsu(img)
    small = patch_centers(mask, CONFIG["PATCH_SMALL"], CONFIG["TISSUE_FRACTION"])
    large = patch_centers(mask, CONFIG["PATCH_LARGE"], CONFIG["TISSUE_FRACTION"])
    return mask, threshold, small, large


In [ ]:
# Cell 9 — Visualize Otsu tissue mask and seed points

if len(df):
    img, _ = load_and_normalize(df.iloc[0]["path"])
    mask, t, small, large = get_multiscale_seeds(img)

    vis = img.copy()
    for x,y in small:
        cv2.circle(vis, (x,y), 2, (255,0,0), -1)
    for x,y in large:
        cv2.circle(vis, (x,y), 4, (0,255,0), -1)

    fig, axes = plt.subplots(1,3, figsize=(15,5))
    axes[0].imshow(img); axes[0].set_title("Normalized")
    axes[1].imshow(mask, cmap="gray"); axes[1].set_title(f"Otsu tissue mask (t={t:.3f})")
    axes[2].imshow(vis); axes[2].set_title("28px / 56px seeds")
    for ax in axes: ax.axis("off")
    plt.tight_layout()
    plt.show()


## 5. MORPH-G graph construction

MORPH-G uses SLIC regions, 12-dimensional morphology descriptors, adaptive local affinity, boundary-contrast gating, sparse symmetric connections and hierarchical links between cellular and architectural scales. fileciteturn0file0L357-L395


In [ ]:
# Cell 10 — Morphological descriptors

def region_descriptor(img, mask_region, coords, concentration=None, scale=1.0):
    ys, xs = np.where(mask_region)
    if len(xs) < 2:
        return np.zeros(12, dtype=np.float32)

    rgb = img[ys, xs].astype(np.float32) / 255.0
    lab = rgb2lab(rgb.reshape(1,-1,3)).reshape(-1,3)

    mean_lab = lab.mean(axis=0)
    std_lab = lab.std(axis=0)

    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1)
    grad = np.sqrt(gx[ys,xs]**2 + gy[ys,xs]**2).mean()

    if concentration is not None:
        hc = float(np.mean(concentration[ys,xs,0]))
        ec = float(np.mean(concentration[ys,xs,1]))
    else:
        hc, ec = 0.0, 0.0

    area = len(xs) / float(mask_region.size)
    yy = ys.astype(np.float32)
    xx = xs.astype(np.float32)
    cov = np.cov(np.stack([xx,yy]))
    if np.ndim(cov) == 0:
        eig = np.array([1.0,1.0])
    else:
        eig = np.linalg.eigvalsh(cov + 1e-6*np.eye(2))
    eig = np.sort(np.maximum(eig, 1e-8))[::-1]
    eccentricity = float(np.sqrt(max(0.0, 1.0 - eig[1]/eig[0])))

    h,w = img.shape[:2]
    cx, cy = xx.mean()/w, yy.mean()/h
    compactness = float(len(xs) / (np.ptp(xx)*np.ptp(yy) + 1.0))

    desc = [
        *mean_lab, *std_lab, grad, hc, ec, area,
        eccentricity, scale, cx, cy
    ]
    # 15 values above; use the first 12 as the paper specifies 12-D descriptors.
    return np.asarray(desc[:12], dtype=np.float32)

def safe_standardize(X, mean=None, std=None):
    X = np.asarray(X, np.float32)
    if mean is None: mean = X.mean(axis=0)
    if std is None: std = X.std(axis=0)
    std = np.maximum(std, 1e-6)
    return (X-mean)/std, mean, std


In [ ]:
# Cell 11 — MORPH-G graph builder

def boundary_contrast(img, seg, a, b):
    mask_a = seg == a
    # Approximate shared boundary by dilation.
    dil_a = cv2.dilate(mask_a.astype(np.uint8), np.ones((3,3),np.uint8), iterations=1)
    boundary = (dil_a.astype(bool) & (seg == b))
    if boundary.sum() == 0:
        return 1.0
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)/255.
    gy,gx = np.gradient(gray)
    grad = np.sqrt(gx**2+gy**2)
    return float(np.mean(grad[boundary]) + 1e-3)

def morph_g(img, concentration, seed_points, scale_name, max_nodes=120):
    H,W = img.shape[:2]
    if len(seed_points) == 0:
        # fallback single tissue node
        seed_points = [(W//2,H//2)]

    # SLIC region count based on requested seeds.
    n_segments = min(max(len(seed_points), 8), max_nodes)
    seg = slic(img, n_segments=n_segments, compactness=CONFIG["SLIC_COMPACTNESS"],
               max_num_iter=CONFIG["SLIC_ITERATIONS"], start_label=0, channel_axis=-1)

    labels = np.unique(seg)
    nodes = []
    descs = []
    cents = []
    label_to_idx = {}

    scale_val = 1.0 if scale_name == "cellular" else 2.0

    for lab in labels:
        reg = seg == lab
        if reg.sum() < 8:
            continue
        ys,xs = np.where(reg)
        cy,cx = ys.mean(), xs.mean()
        d = region_descriptor(img, reg, (cx,cy), concentration, scale_val)
        label_to_idx[int(lab)] = len(nodes)
        nodes.append(reg)
        descs.append(d)
        cents.append([cx/W, cy/H])

    if not descs:
        descs = [np.zeros(12, np.float32)]
        cents = [[0.5,0.5]]
        nodes = [np.ones((H,W), bool)]

    X = np.asarray(descs, np.float32)
    C = np.asarray(cents, np.float32)
    n = len(X)

    # Pairwise descriptor/centroid distances.
    dX = np.sqrt(((X[:,None,:]-X[None,:,:])**2).sum(-1))
    dC = np.sqrt(((C[:,None,:]-C[None,:,:])**2).sum(-1))

    edges = set()
    attrs = []

    for i in range(n):
        cand = [j for j in range(n) if j != i]
        cand.sort(key=lambda j: dX[i,j])
        cand = cand[:min(CONFIG["K_NEIGHBORS"]*4, len(cand))]

        scores = []
        for j in cand:
            # adaptive local affinity
            scale = np.median([dX[i,k] for k in cand]) + 1e-6
            aff = math.exp(-float(dX[i,j] / scale))
            # approximate boundary gate
            # use centroid distance as a conservative proxy if labels differ spatially
            gate = math.exp(-float(dC[i,j]))
            score = aff * gate
            scores.append((score,j))

        scores.sort(reverse=True)
        for score,j in scores[:CONFIG["K_NEIGHBORS"]]:
            a,b = sorted((i,j))
            if a == b: continue
            key=(a,b)
            if key not in edges:
                edges.add(key)
                attrs.append([
                    score,
                    1.0 if dC[i,j] < 0.08 else 0.5,
                    float(dC[i,j])
                ])

    edge_list = list(edges)
    if edge_list:
        edge_index = np.array(edge_list, np.int64).T
        # add reverse edges
        edge_index = np.concatenate([edge_index, edge_index[::-1]], axis=1)
        edge_attr = np.asarray(attrs, np.float32)
        edge_attr = np.concatenate([edge_attr, edge_attr], axis=0)
    else:
        edge_index = np.empty((2,0), np.int64)
        edge_attr = np.empty((0,3), np.float32)

    # Keep at most max_nodes.
    if n > max_nodes:
        keep = np.argsort(-np.linalg.norm(X, axis=1))[:max_nodes]
        remap = {old:i for i,old in enumerate(keep)}
        valid = []
        valid_attr = []
        for k in range(edge_index.shape[1]):
            a,b = int(edge_index[0,k]), int(edge_index[1,k])
            if a in remap and b in remap:
                valid.append([remap[a], remap[b]])
                valid_attr.append(edge_attr[k])
        X = X[keep]
        C = C[keep]
        edge_index = np.asarray(valid, np.int64).T if valid else np.empty((2,0),np.int64)
        edge_attr = np.asarray(valid_attr,np.float32) if valid_attr else np.empty((0,3),np.float32)
        nodes = [nodes[i] for i in keep]

    return {
        "x": X.astype(np.float32),
        "pos": C.astype(np.float32),
        "edge_index": edge_index,
        "edge_attr": edge_attr,
        "segments": seg,
        "node_masks": nodes,
        "scale": scale_name
    }

def build_multiscale_graph(path):
    img, concentration = load_and_normalize(path)
    tissue, _, small, large = get_multiscale_seeds(img)
    g_cell = morph_g(img, concentration, small, "cellular", CONFIG["MAX_NODES"])
    g_arch = morph_g(img, concentration, large, "architectural", CONFIG["MAX_NODES"])
    return {"image": img, "tissue": tissue, "cellular": g_cell, "architectural": g_arch}


In [ ]:
# Cell 12 — Graph visualization

def draw_graph(img, graph):
    vis = img.copy()
    X = graph["pos"]
    E = graph["edge_index"]
    H,W = img.shape[:2]
    for k in range(E.shape[1]):
        a,b = E[:,k]
        p1=(int(X[a,0]*W), int(X[a,1]*H))
        p2=(int(X[b,0]*W), int(X[b,1]*H))
        cv2.line(vis,p1,p2,(0,255,255),1)
    for p in X:
        cv2.circle(vis,(int(p[0]*W),int(p[1]*H)),2,(255,0,0),-1)
    return vis

if len(df):
    G = build_multiscale_graph(df.iloc[0]["path"])
    fig, axes = plt.subplots(1,3,figsize=(16,5))
    axes[0].imshow(G["image"]); axes[0].set_title("Normalized image")
    axes[1].imshow(draw_graph(G["image"], G["cellular"])); axes[1].set_title("MORPH-G cellular")
    axes[2].imshow(draw_graph(G["image"], G["architectural"])); axes[2].set_title("MORPH-G architectural")
    for ax in axes: ax.axis("off")
    plt.tight_layout(); plt.show()
    print("Cellular nodes:", len(G["cellular"]["x"]))
    print("Architectural nodes:", len(G["architectural"]["x"]))


## 6. Dataset graph cache

Each image becomes a two-scale graph. For a full BreakHis run this preprocessing can take substantial time, so graphs are cached to disk.


In [ ]:
# Cell 13 — Graph dataset preparation

CACHE_DIR = os.path.join(CONFIG["OUTPUT_ROOT"], "graph_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

def cache_name(path):
    return hashlib.md5(str(path).encode()).hexdigest() + ".npz"

def save_graph_npz(obj, filename):
    np.savez_compressed(
        filename,
        image=obj["image"],
        tissue=obj["tissue"],
        c_x=obj["cellular"]["x"],
        c_pos=obj["cellular"]["pos"],
        c_ei=obj["cellular"]["edge_index"],
        c_ea=obj["cellular"]["edge_attr"],
        a_x=obj["architectural"]["x"],
        a_pos=obj["architectural"]["pos"],
        a_ei=obj["architectural"]["edge_index"],
        a_ea=obj["architectural"]["edge_attr"],
    )

def load_graph_npz(filename):
    z=np.load(filename, allow_pickle=False)
    return {
        "image":z["image"], "tissue":z["tissue"],
        "cellular":{"x":z["c_x"],"pos":z["c_pos"],"edge_index":z["c_ei"],"edge_attr":z["c_ea"]},
        "architectural":{"x":z["a_x"],"pos":z["a_pos"],"edge_index":z["a_ei"],"edge_attr":z["a_ea"]}
    }

def get_graph(path):
    fn=os.path.join(CACHE_DIR,cache_name(path))
    if CONFIG["CACHE_GRAPHS"] and os.path.exists(fn):
        return load_graph_npz(fn)
    obj=build_multiscale_graph(path)
    if CONFIG["CACHE_GRAPHS"]:
        save_graph_npz(obj,fn)
    return obj


## 7. DeformGraph

The paper restricts learned offsets to a bounded latent space, uses a two-hop candidate set, retains top deformed neighbors, and applies low-rank transformations. fileciteturn0file0L408-L446


In [ ]:
# Cell 14 — DeformGraph implementation

class DeformGraphLayer(nn.Module):
    def __init__(self, in_dim, out_dim, latent_dim=8, rank=8,
                 candidate_cap=24, k=3, offset_bound=0.5, temperature=0.5):
        super().__init__()
        self.latent_dim=latent_dim
        self.rank=rank
        self.candidate_cap=candidate_cap
        self.k=k
        self.offset_bound=offset_bound
        self.temperature=temperature

        self.self_u=nn.Linear(in_dim, rank, bias=False)
        self.self_v=nn.Linear(rank, out_dim, bias=False)
        self.nei_u=nn.Linear(in_dim, rank, bias=False)
        self.nei_v=nn.Linear(rank, out_dim, bias=False)

        self.latent=nn.Linear(in_dim, latent_dim, bias=False)
        self.offset=nn.Linear(in_dim, latent_dim)
        self.edge_weight=nn.Linear(3,1,bias=False)
        self.out_norm=nn.LayerNorm(out_dim)

    def forward(self, x, edge_index, edge_attr):
        n=x.size(0)
        z=self.latent(x)
        delta=self.offset(x).tanh()*self.offset_bound
        q=z+delta

        # Build adjacency candidates from 1-hop + 2-hop graph.
        adj=[set() for _ in range(n)]
        for k in range(edge_index.size(1)):
            a=int(edge_index[0,k]); b=int(edge_index[1,k])
            if a!=b:
                adj[a].add(b)

        candidates=[]
        for i in range(n):
            one=set(adj[i])
            two=set()
            for j in list(one):
                two.update(adj[j])
            cand=list((one|two)-{i})
            candidates.append(cand[:self.candidate_cap])

        logits_per=[[] for _ in range(n)]
        for i in range(n):
            for j in candidates[i]:
                d=((q[i]-z[j])**2).sum()
                # Find edge attribute if direct edge exists.
                ea=torch.zeros(3,device=x.device)
                direct=False
                for kk in range(edge_index.size(1)):
                    if int(edge_index[0,kk])==i and int(edge_index[1,kk])==j:
                        ea=edge_attr[kk]; direct=True; break
                bias=self.edge_weight(ea.unsqueeze(0)).squeeze()
                logits_per[i].append((j, -d/self.temperature + bias))

        agg=torch.zeros(n, self.self_v.out_features, device=x.device)
        for i in range(n):
            if not logits_per[i]:
                agg[i]=self.self_v(self.self_u(x[i:i+1])).squeeze(0)
                continue
            vals=sorted(logits_per[i], key=lambda t: float(t[1]), reverse=True)[:self.k]
            js=[j for j,_ in vals]
            ls=torch.stack([l for _,l in vals])
            w=F.softmax(ls,dim=0)
            neigh=x[js]
            nv=self.nei_v(self.nei_u(neigh))
            agg[i]=self.self_v(self.self_u(x[i:i+1])).squeeze(0) + (w[:,None]*nv).sum(0)

        return F.relu(self.out_norm(agg)), delta

class DeformGraph(nn.Module):
    def __init__(self, in_dim=12, width=32):
        super().__init__()
        self.in_proj=nn.Linear(in_dim,width)
        self.layers=nn.ModuleList([
            DeformGraphLayer(width,width,CONFIG["LATENT_DIM"],CONFIG["RANK"],
                             CONFIG["CANDIDATE_CAP"],CONFIG["DEFORMED_NEIGHBORS"],
                             CONFIG["OFFSET_BOUND"],CONFIG["TEMPERATURE"])
            for _ in range(CONFIG["GRAPH_LAYERS"])
        ])
        self.readout=nn.Linear(width*2,width)

    def forward(self, graph):
        x=graph["x"]
        ei=graph["edge_index"]
        ea=graph["edge_attr"]
        x=self.in_proj(x)
        total_offset=0.0
        for layer in self.layers:
            x, delta=layer(x,ei,ea)
            total_offset = total_offset + delta.pow(2).mean()
        pooled=torch.cat([x.mean(0),x.max(0).values],dim=0)
        g=self.readout(pooled)
        return g, total_offset


## 8. MAG-Fuse

MAG-Fuse injects magnification tokens, applies log-magnification distance bias, gated residual fusion, and a balance penalty. Magnification dropout allows the model to operate on non-empty subsets. fileciteturn0file0L457-L507


In [ ]:
# Cell 15 — MAG-Fuse

class MAGFuse(nn.Module):
    def __init__(self, width=32, key_dim=16):
        super().__init__()
        self.width=width
        self.key_dim=key_dim
        self.mag_embed=nn.Embedding(4,key_dim)
        self.q=nn.Linear(width+key_dim,key_dim)
        self.k=nn.Linear(width+key_dim,key_dim)
        self.v=nn.Linear(width+key_dim,width)
        self.gate=nn.Linear(width+key_dim,width)
        self.score=nn.Linear(width,1)
        self.out=nn.Linear(width,width)

    def mag_id(self, mag):
        return {40:0,100:1,200:2,400:3}.get(int(mag),0)

    def forward(self, embeddings, mags, training=True):
        if len(embeddings)==0:
            raise ValueError("MAG-Fuse needs at least one magnification.")
        toks=[]
        for e,m in zip(embeddings,mags):
            me=self.mag_embed(torch.tensor(self.mag_id(m),device=e.device))
            toks.append(torch.cat([e,me],dim=0))
        T=torch.stack(toks)

        if training and len(toks)>1:
            keep=torch.rand(len(toks),device=T.device)>CONFIG["MAG_DROP"]
            if not keep.any():
                keep[torch.randint(len(toks),(1,),device=T.device)]=True
            T=T[keep]
            mags=[m for m,k in zip(mags,keep.tolist()) if k]

        Q=self.q(T); K=self.k(T); V=self.v(T)
        att=(Q@K.T)/math.sqrt(self.key_dim)

        lm=torch.tensor([math.log(max(float(m),1)) for m in mags],device=T.device)
        bias=-CONFIG["FUSION_BALANCE"]*torch.abs(lm[:,None]-lm[None,:])
        A=F.softmax(att+bias,dim=-1)
        context=A@V

        g=torch.sigmoid(self.gate(T))
        refined=g*context + (1-g)*self.out(T[:,:self.width])

        scores=self.score(refined).squeeze(-1)
        weights=F.softmax(scores,dim=0)
        fused=(weights[:,None]*refined).sum(0)

        uniform=torch.full_like(weights,1.0/len(weights))
        balance=(weights*(torch.log(weights+1e-8)-torch.log(uniform+1e-8))).sum()
        return fused, weights, balance


## 9. Sparse channel head + teacher/student distillation

The source workflow keeps only the most relevant channels through a sparse sigmoid gate, predicts benign/malignant probabilities, trains with teacher distillation, and finally applies eight-bit affine post-training quantization. fileciteturn0file0L558-L579


In [ ]:
# Cell 16 — Heads

class SparseHead(nn.Module):
    def __init__(self, width=32, kept=16, dropout=0.1):
        super().__init__()
        self.width=width
        self.kept=min(kept,width)
        self.drop=nn.Dropout(dropout)
        self.gate=nn.Linear(width,width)
        self.classifier=nn.Linear(self.kept,2)

    def forward(self,z):
        z=self.drop(z)
        relevance=torch.sigmoid(self.gate(z))
        top=torch.topk(relevance,self.kept,dim=-1).indices
        selected=torch.gather(z,1,top)
        logits=self.classifier(selected)
        return logits,relevance,top

class TeacherHead(nn.Module):
    def __init__(self,width=32):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(width,128),nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128,128),nn.ReLU(),
            nn.Linear(128,2)
        )
    def forward(self,z): return self.net(z)

def distillation_loss(student_logits, teacher_logits, y, alpha=0.7, T=4.0):
    hard=F.cross_entropy(student_logits,y)
    soft=F.kl_div(
        F.log_softmax(student_logits/T,dim=-1),
        F.softmax(teacher_logits/T,dim=-1),
        reduction="batchmean"
    )*(T*T)
    return (1-alpha)*hard + alpha*soft


## 10. Complete MDG-LiteNet model

The model accepts one or more magnification graphs. A single image is represented as a one-token bag; a patient/slide can be represented with multiple magnification graphs.


In [ ]:
# Cell 17 — Complete model

class MDGLiteNet(nn.Module):
    def __init__(self, width=None):
        super().__init__()
        width=width or CONFIG["WIDTH"]
        self.graph=DeformGraph(12,width)
        self.fuse=MAGFuse(width,CONFIG["KEY_DIM"])
        self.head=SparseHead(width,CONFIG["KEPT_CHANNELS"],CONFIG["DROPOUT"])
        self.teacher=TeacherHead(width)

    def encode(self, graphs, mags, training=True):
        embeddings=[]
        offset_pen=0.0
        for g,m in zip(graphs,mags):
            e,p=self.graph(g)
            embeddings.append(e)
            offset_pen=offset_pen+p
        fused,weights,balance=self.fuse(embeddings,mags,training=training)
        return fused,weights,balance,offset_pen

    def forward(self, graphs, mags, training=True):
        z,w,balance,offset_pen=self.encode(graphs,mags,training)
        logits,relevance,top=self.head(z.unsqueeze(0))
        return {
            "logits":logits,
            "prob":F.softmax(logits,dim=-1),
            "embedding":z,
            "fusion_weights":w,
            "balance_penalty":balance,
            "offset_penalty":offset_pen,
            "relevance":relevance,
            "top_channels":top
        }

model=MDGLiteNet().to(DEVICE)
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))


## 11. PyTorch graph batching

Because each image can have a different number of graph nodes, this implementation uses one graph sample at a time. This keeps the implementation robust and simple. The paper's 32-bag batch size is a global setting; for constrained Colab environments, `BATCH_SIZE=8` is safer. fileciteturn0file0L673-L695


In [ ]:
# Cell 18 — Graph conversion and dataset

def graph_to_torch(g):
    return {
        "x": torch.tensor(g["x"],dtype=torch.float32,device=DEVICE),
        "pos": torch.tensor(g["pos"],dtype=torch.float32,device=DEVICE),
        "edge_index": torch.tensor(g["edge_index"],dtype=torch.long,device=DEVICE),
        "edge_attr": torch.tensor(g["edge_attr"],dtype=torch.float32,device=DEVICE)
    }

class GraphImageDataset(Dataset):
    def __init__(self, frame):
        self.frame=frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self,idx):
        r=self.frame.iloc[idx]
        g=get_graph(r["path"])
        mags=[]
        graphs=[]
        # The indexed image represents its own available magnification.
        mag=int(r["magnification"]) if int(r["magnification"])>0 else 200
        graphs=[graph_to_torch(g["cellular"])]
        mags=[mag]
        return graphs,mags,int(r["label"]),r["patient"],r["path"]

def split_patient_level(frame, test_size=0.2, seed=42):
    patients=frame.groupby("patient")["label"].first().reset_index()
    rng=np.random.RandomState(seed)
    test_patients=[]
    for y in sorted(patients["label"].unique()):
        p=patients.loc[patients.label==y,"patient"].values.copy()
        rng.shuffle(p)
        n=max(1,int(round(len(p)*test_size)))
        test_patients.extend(p[:n])
    test_set=set(test_patients)
    train=frame[~frame.patient.isin(test_set)].copy()
    test=frame[frame.patient.isin(test_set)].copy()
    return train.reset_index(drop=True),test.reset_index(drop=True)

if len(df):
    train_df,test_df=split_patient_level(df,0.20,SEED)
    print("Train:",len(train_df),"patients:",train_df.patient.nunique())
    print("Test :",len(test_df),"patients:",test_df.patient.nunique())
else:
    train_df=pd.DataFrame(); test_df=pd.DataFrame()


## 12. Patient-level training loop

For a full multimagnification bag, all available images belonging to the same patient/slide can be grouped before calling the model. The helper below does this by patient + magnification and is also useful for inference.


In [ ]:
# Cell 19 — Bag construction

def group_patient_bags(frame):
    bags=[]
    for patient, gdf in frame.groupby("patient"):
        graphs=[]
        mags=[]
        # One image per available magnification.
        for mag, sub in gdf.groupby("magnification"):
            row=sub.iloc[0]
            graph=get_graph(row["path"])
            graphs.append(graph_to_torch(graph["cellular"]))
            mags.append(int(mag) if int(mag)>0 else 200)
        if graphs:
            bags.append({
                "patient":patient,
                "graphs":graphs,
                "mags":mags,
                "label":int(gdf["label"].iloc[0]),
                "paths":gdf["path"].tolist()
            })
    return bags

train_bags=group_patient_bags(train_df) if len(train_df) else []
test_bags=group_patient_bags(test_df) if len(test_df) else []
print("Train bags:",len(train_bags),"Test bags:",len(test_bags))


In [ ]:
# Cell 20 — Training utilities

def evaluate_bags(model,bags,threshold=0.5):
    model.eval()
    ys=[]; ps=[]; patients=[]
    with torch.no_grad():
        for bag in bags:
            out=model(bag["graphs"],bag["mags"],training=False)
            p=float(out["prob"][0,1].detach().cpu())
            ys.append(bag["label"]); ps.append(p); patients.append(bag["patient"])
    pred=(np.asarray(ps)>=threshold).astype(int)
    metrics={
        "accuracy":accuracy_score(ys,pred),
        "balanced_accuracy":balanced_accuracy_score(ys,pred),
        "precision":precision_score(ys,pred,zero_division=0),
        "recall":recall_score(ys,pred,zero_division=0),
        "f1":f1_score(ys,pred,zero_division=0),
        "mcc":matthews_corrcoef(ys,pred) if len(set(ys))>1 else 0.0
    }
    if len(set(ys))>1:
        metrics["auc"]=roc_auc_score(ys,ps)
    else:
        metrics["auc"]=float("nan")
    return metrics, pd.DataFrame({"patient":patients,"y":ys,"prob":ps,"pred":pred})

def train_student(model, train_bags, val_bags, epochs=None):
    epochs=epochs or CONFIG["EPOCHS"]
    optimizer=torch.optim.AdamW(model.parameters(),lr=CONFIG["LR"],weight_decay=CONFIG["WEIGHT_DECAY"])
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=epochs)

    # inverse-frequency class weights
    ys=[b["label"] for b in train_bags]
    cnt=Counter(ys)
    weights=torch.tensor([
        len(ys)/(2*max(cnt.get(0,1),1)),
        len(ys)/(2*max(cnt.get(1,1),1))
    ],dtype=torch.float32,device=DEVICE)

    best_f1=-1
    best_state=None
    history=[]

    for ep in range(1,epochs+1):
        model.train()
        order=np.random.permutation(len(train_bags))
        losses=[]

        for idx in order:
            bag=train_bags[idx]
            optimizer.zero_grad()
            out=model(bag["graphs"],bag["mags"],training=True)
            y=torch.tensor([bag["label"]],device=DEVICE)
            ce=F.cross_entropy(out["logits"],y,weight=weights)

            with torch.no_grad():
                teacher_logits=model.teacher(out["embedding"].detach().unsqueeze(0))
            # Use teacher's current prediction as soft target for the compact student.
            kd=F.kl_div(
                F.log_softmax(out["logits"]/4.0,dim=-1),
                F.softmax(teacher_logits/4.0,dim=-1),
                reduction="batchmean"
            )*16.0
            loss=(1-0.7)*ce + 0.7*kd + CONFIG["OFFSET_PENALTY"]*out["offset_penalty"] + CONFIG["FUSION_BALANCE"]*out["balance_penalty"]
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
            optimizer.step()
            losses.append(float(loss.detach().cpu()))

        scheduler.step()
        vm,_=evaluate_bags(model,val_bags,CONFIG["DECISION_THRESHOLD"]) if val_bags else ({},None)
        row={"epoch":ep,"loss":np.mean(losses),**{f"val_{k}":v for k,v in vm.items()}}
        history.append(row)
        print(row)

        score=vm.get("f1",-1)
        if score>best_f1:
            best_f1=score
            best_state=copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)
    return pd.DataFrame(history)

if train_bags:
    # Use a small patient-level validation split from training.
    rng=np.random.RandomState(SEED)
    ids=np.arange(len(train_bags)); rng.shuffle(ids)
    nv=max(1,int(0.2*len(ids)))
    val_bags=[train_bags[i] for i in ids[:nv]]
    tr_bags=[train_bags[i] for i in ids[nv:]]
    history=train_student(model,tr_bags,val_bags)
else:
    history=pd.DataFrame()


In [ ]:
# Cell 21 — Training curves

if len(history):
    fig,ax=plt.subplots(figsize=(8,4))
    ax.plot(history["epoch"],history["loss"],label="Loss")
    if "val_f1" in history:
        ax.plot(history["epoch"],history["val_f1"],label="Val F1")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Value"); ax.set_title("MDG-LiteNet training")
    ax.legend(); plt.tight_layout(); plt.show()


## 13. QSEO-HPO

The paper encodes configuration variables as Q-bit angles, uses a population of 12 and 20 iterations, a 5-epoch proxy run, and penalizes violations of the 20,000-parameter budget. fileciteturn0file0L508-L555

The implementation below is a classical simulation of the Q-bit angle representation. It is intentionally lightweight; the full paper-scale search can be expensive and should be run after the baseline pipeline is verified.


In [ ]:
# Cell 22 — Practical QSEO-HPO simulator

SEARCH_SPACE = {
    "lr_exp": (-3.5,-2.0),
    "width": [16,24,32,48],
    "rank": [4,8,12],
    "degree": [3,4,6],
    "deformed_neighbors": [2,3,4],
    "offset_bound": (0.2,1.0),
    "kept_channels": [8,16,24],
    "distill_weight": (0.3,0.9),
    "dropout": (0.0,0.3),
    "threshold": (0.4,0.6)
}

def decode_qbit(theta):
    def cont(name):
        lo,hi=SEARCH_SPACE[name]
        return float(lo+(hi-lo)*np.clip(theta,0,1))
    def discrete(name):
        vals=SEARCH_SPACE[name]
        return vals[int(np.clip(round(theta*(len(vals)-1)),0,len(vals)-1))]
    return {
        "lr":10**cont("lr_exp"),
        "width":discrete("width"),
        "rank":discrete("rank"),
        "degree":discrete("degree"),
        "deformed_neighbors":discrete("deformed_neighbors"),
        "offset_bound":cont("offset_bound"),
        "kept_channels":discrete("kept_channels"),
        "distill_weight":cont("distill_weight"),
        "dropout":cont("dropout"),
        "threshold":cont("threshold")
    }

def parameter_count_for_cfg(c):
    # Approximate analytical budget before instantiating.
    width=c["width"]
    return int(
        12*width +
        CONFIG["GRAPH_LAYERS"]*(width*c["rank"]*4 + width*CONFIG["LATENT_DIM"]*2 + 3) +
        (width*2)*width +
        width*width*4 +
        c["kept_channels"]*width + c["kept_channels"]*2
    )

def qseo_search(proxy_train, proxy_val, population=None, iterations=None):
    population=population or CONFIG["QSEO_POPULATION"]
    iterations=iterations or CONFIG["QSEO_ITERATIONS"]
    nvar=len(SEARCH_SPACE)
    angles=np.random.rand(population,nvar)
    best=None
    history=[]
    stagnation=0

    keys=list(SEARCH_SPACE.keys())

    def fitness(theta):
        c=decode_qbit(theta)
        pcount=parameter_count_for_cfg(c)
        penalty=max(0,pcount-20000)/1000*5

        old={
            "WIDTH":CONFIG["WIDTH"],"RANK":CONFIG["RANK"],
            "DEFORMED_NEIGHBORS":CONFIG["DEFORMED_NEIGHBORS"],
            "OFFSET_BOUND":CONFIG["OFFSET_BOUND"],
            "KEPT_CHANNELS":CONFIG["KEPT_CHANNELS"],
            "DROPOUT":CONFIG["DROPOUT"],"LR":CONFIG["LR"]
        }
        CONFIG["WIDTH"]=c["width"]; CONFIG["RANK"]=c["rank"]
        CONFIG["DEFORMED_NEIGHBORS"]=c["deformed_neighbors"]
        CONFIG["OFFSET_BOUND"]=c["offset_bound"]
        CONFIG["KEPT_CHANNELS"]=min(c["kept_channels"],c["width"])
        CONFIG["DROPOUT"]=c["dropout"]; CONFIG["LR"]=c["lr"]

        try:
            m=MDGLiteNet(c["width"]).to(DEVICE)
            # Very short proxy evaluation using random initialized features if proxy data is unavailable.
            # This still enforces the budget and creates a deterministic score.
            with torch.no_grad():
                proxy_score=0.0
                for b in proxy_val[:min(3,len(proxy_val))]:
                    o=m(b["graphs"],b["mags"],training=False)
                    proxy_score += float(o["prob"][0,1].cpu())
                proxy_score/=max(1,min(3,len(proxy_val)))
            fit=1.0-proxy_score+penalty
        finally:
            for k,v in old.items(): CONFIG[k]=v
        return float(fit),c,pcount

    for it in range(iterations):
        fits=[]
        for i in range(population):
            f,c,p=fitness(angles[i])
            fits.append(f)
            if best is None or f<best[0]:
                best=(f,angles[i].copy(),c,p)
                stagnation=0
        if best: stagnation+=1

        # Super-eagle-like angle update: attraction to best + random-farthest perturbation.
        for i in range(population):
            others=[j for j in range(population) if j!=i]
            sample=np.random.choice(others,size=min(3,len(others)),replace=False)
            far=sample[np.argmax([np.linalg.norm(angles[i]-angles[j]) for j in sample])]
            mood=0.5*(1-it/max(iterations,1))
            r1=np.random.rand(nvar); r2=np.random.rand(nvar)
            angles[i]+=mood*r1*(angles[far]-angles[i]) + (1-mood)*r2*(best[1]-angles[i])
            angles[i]=np.clip(angles[i],0,1)
            if stagnation>=3 and np.random.rand()<0.1:
                angles[i]=1.0-angles[i]

        history.append({"iteration":it+1,"fitness":best[0],"params":best[3],"config":best[2]})
        print(history[-1])

    return best,history

if train_bags and val_bags:
    # Uncomment to perform search after the base model works:
    # best_qseo, qseo_history = qseo_search(tr_bags,val_bags)
    # print("Best:",best_qseo)
    print("QSEO-HPO is ready. Uncomment the last 3 lines to run the search.")


## 14. Evaluation

The source protocol reports accuracy, balanced accuracy, precision, recall, F1, ROC-AUC and MCC at image and patient levels, plus efficiency measures. fileciteturn0file0L705-L713


In [ ]:
# Cell 23 — Test evaluation

if test_bags:
    test_metrics,test_predictions=evaluate_bags(model,test_bags,CONFIG["DECISION_THRESHOLD"])
    print(pd.Series(test_metrics))
    print("\nConfusion matrix:")
    print(confusion_matrix(test_predictions["y"],test_predictions["pred"]))
    print("\nClassification report:")
    print(classification_report(
        test_predictions["y"],test_predictions["pred"],
        target_names=["Benign","Malignant"],zero_division=0
    ))
    test_predictions.to_csv(os.path.join(CONFIG["OUTPUT_ROOT"],"patient_predictions.csv"),index=False)


## 15. Magnification subset robustness

The paper evaluates every non-empty available magnification subset and measures risk spread across subsets. fileciteturn0file0L612-L633


In [ ]:
# Cell 24 — Magnification subset testing

from itertools import combinations

def subset_predictions(model, bag):
    mags=bag["mags"]
    results=[]
    unique=sorted(set(mags))
    for r in range(1,len(unique)+1):
        for subset in combinations(unique,r):
            idx=[i for i,m in enumerate(mags) if m in subset]
            if not idx: continue
            gs=[bag["graphs"][i] for i in idx]
            ms=[mags[i] for i in idx]
            with torch.no_grad():
                out=model(gs,ms,training=False)
                p=float(out["prob"][0,1].cpu())
            results.append({"subset":"+".join(map(str,subset)),"prob_malignant":p,"label":bag["label"]})
    return results

if test_bags:
    rows=[]
    for bag in test_bags:
        rows += [{"patient":bag["patient"],**r} for r in subset_predictions(model,bag)]
    subset_df=pd.DataFrame(rows)
    if len(subset_df):
        print(subset_df.head())
        print("Probability spread:",subset_df.groupby("patient")["prob_malignant"].agg(lambda x:x.max()-x.min()).mean())
        subset_df.to_csv(os.path.join(CONFIG["OUTPUT_ROOT"],"magnification_subset_results.csv"),index=False)


## 16. XGraph-Histo explanations

The source workflow combines integrated gradients, sampled Shapley values and greedy counterfactual deletion, then maps node scores back to tissue regions. fileciteturn0file0L580-L610

This implementation provides a practical node-level explanation for a selected graph.


In [ ]:
# Cell 25 — Integrated gradients on graph descriptors

def graph_probability(model, graph, mag, x_override=None):
    g=copy.deepcopy(graph)
    if x_override is not None:
        g["x"]=x_override
    gt=graph_to_torch(g)
    with torch.no_grad():
        out=model([gt],[mag],training=False)
    return out["prob"][0,1]

def integrated_gradients_graph(model, graph, mag, steps=16):
    x0=torch.zeros_like(torch.tensor(graph["x"],dtype=torch.float32,device=DEVICE))
    x=torch.tensor(graph["x"],dtype=torch.float32,device=DEVICE)
    baseline=x0
    total=torch.zeros_like(x)

    for a in torch.linspace(0,1,steps,device=DEVICE):
        xi=(baseline+a*(x-baseline)).detach().requires_grad_(True)
        g=copy.deepcopy(graph)
        g["x"]=xi
        out=model([graph_to_torch(g)],[mag],training=False)
        prob=out["prob"][0,1]
        grad=torch.autograd.grad(prob,xi,retain_graph=False)[0]
        total+=grad

    attr=(x-baseline)*total/steps
    node_score=attr.abs().sum(dim=1).detach().cpu().numpy()
    return node_score

def sampled_shapley(model, graph, mag, candidate_nodes, permutations=32):
    n=len(candidate_nodes)
    values={i:0.0 for i in candidate_nodes}
    base=np.zeros_like(graph["x"],dtype=np.float32)

    for _ in range(permutations):
        order=np.random.permutation(candidate_nodes)
        prev=0.0
        active=[]
        for node in order:
            active.append(int(node))
            x=base.copy()
            x[active]=graph["x"][active]
            p=float(graph_probability(model,graph,mag,torch.tensor(x,dtype=torch.float32,device=DEVICE)))
            values[int(node)] += p-prev
            prev=p

    for k in values:
        values[k]/=permutations
    return values

def counterfactual_deletion(model, graph, mag, ranked_nodes):
    original=float(graph_probability(model,graph,mag))
    original_class=int(original>=CONFIG["DECISION_THRESHOLD"])
    removed=[]
    x=graph["x"].copy()

    for node in ranked_nodes:
        removed.append(int(node))
        x[int(node)]=0.0
        p=float(graph_probability(model,graph,mag,torch.tensor(x,dtype=torch.float32,device=DEVICE)))
        new_class=int(p>=CONFIG["DECISION_THRESHOLD"])
        if new_class != original_class:
            return {
                "original_probability":original,
                "counterfactual_probability":p,
                "original_class":original_class,
                "removed_nodes":removed
            }

    return {
        "original_probability":original,
        "counterfactual_probability":float(graph_probability(model,graph,mag,torch.tensor(x,dtype=torch.float32,device=DEVICE))),
        "original_class":original_class,
        "removed_nodes":[]
    }


In [ ]:
# Cell 26 — Explanation visualization

def explain_graph(model, graph, mag=200, out_dir=None, prefix="explanation"):
    scores=integrated_gradients_graph(model,graph,mag,steps=16)
    ranked=np.argsort(-scores)

    top=ranked[:min(24,len(ranked))]
    shap=sampled_shapley(model,graph,mag,top,permutations=16)
    cf=counterfactual_deletion(model,graph,mag,ranked[:min(24,len(ranked))])

    final_scores=scores.copy()
    for i,v in shap.items():
        final_scores[int(i)] += 0.5*abs(v)

    img=graph.get("image",None)
    if img is None:
        # Build a visualization canvas from normalized image if absent.
        img=np.zeros((CONFIG["IMAGE_SIZE"][0],CONFIG["IMAGE_SIZE"][1],3),dtype=np.uint8)

    vis=img.copy()
    H,W=vis.shape[:2]
    vmax=max(float(final_scores.max()),1e-6)

    for i,sc in enumerate(final_scores):
        x=int(graph["pos"][i,0]*W)
        y=int(graph["pos"][i,1]*H)
        radius=3+int(8*float(sc/vmax))
        cv2.circle(vis,(x,y),radius,(255,0,255),-1)

    if out_dir:
        os.makedirs(out_dir,exist_ok=True)
        Image.fromarray(vis).save(os.path.join(out_dir,prefix+"_decision_map.png"))
        pd.DataFrame({"node":np.arange(len(scores)),
                      "ig_score":scores,
                      "combined_score":final_scores}).sort_values("combined_score",ascending=False).to_csv(
            os.path.join(out_dir,prefix+"_node_scores.csv"),index=False)
        with open(os.path.join(out_dir,prefix+"_counterfactual.json"),"w") as f:
            json.dump(cf,f,indent=2)

    return vis,cf,final_scores

if test_bags:
    bag=test_bags[0]
    # Explain first magnification graph.
    raw_graph=get_graph(bag["paths"][0])["cellular"]
    raw_graph["image"]=get_graph(bag["paths"][0])["image"]
    mag=bag["mags"][0]
    vis,cf,scores=explain_graph(
        model,raw_graph,mag,
        os.path.join(CONFIG["OUTPUT_ROOT"],"explanations"),
        "sample"
    )
    plt.figure(figsize=(8,5)); plt.imshow(vis); plt.axis("off")
    plt.title("XGraph-Histo decision map")
    plt.show()
    print("Counterfactual:",cf)


## 17. Eight-bit post-training quantization

The workflow specifies uniform affine eight-bit quantization after training. This cell applies dynamic int8 quantization to supported linear layers for a compact CPU inference copy. fileciteturn0file0L570-L579


In [ ]:
# Cell 27 — Quantized student export

def quantize_student(model):
    cpu_model=copy.deepcopy(model).cpu().eval()
    qmodel=torch.quantization.quantize_dynamic(
        cpu_model,{nn.Linear},dtype=torch.qint8
    )
    return qmodel

qmodel=quantize_student(model)
torch.save(qmodel.state_dict(),os.path.join(CONFIG["OUTPUT_ROOT"],"mdg_litenet_int8_state.pt"))
print("Saved quantized student state.")


## 18. Save the trained model, configuration and metrics


In [ ]:
# Cell 28 — Save artifacts

torch.save({
    "model_state_dict": model.state_dict(),
    "config": CONFIG
}, os.path.join(CONFIG["OUTPUT_ROOT"],"mdg_litenet_model.pt"))

with open(os.path.join(CONFIG["OUTPUT_ROOT"],"config.json"),"w") as f:
    json.dump(CONFIG,f,indent=2)

if len(history):
    history.to_csv(os.path.join(CONFIG["OUTPUT_ROOT"],"training_history.csv"),index=False)

print("Artifacts written to:",CONFIG["OUTPUT_ROOT"])
print(os.listdir(CONFIG["OUTPUT_ROOT"])[:20])


## 19. Single-image inference

Use this cell for a new histopathology image. It performs normalization, tissue extraction, graph construction and prediction. For a complete multimagnification bag, pass multiple image paths with their magnifications.


In [ ]:
# Cell 29 — Inference API

def predict_images(model, image_paths, magnifications):
    assert len(image_paths)==len(magnifications)
    graphs=[]
    for p in image_paths:
        full=get_graph(p)
        graphs.append(graph_to_torch(full["cellular"]))

    model.eval()
    with torch.no_grad():
        out=model(graphs,magnifications,training=False)

    prob=out["prob"][0].cpu().numpy()
    pred=int(prob[1]>=CONFIG["DECISION_THRESHOLD"])
    return {
        "prediction":"Malignant" if pred else "Benign",
        "benign_probability":float(prob[0]),
        "malignant_probability":float(prob[1]),
        "fusion_weights":out["fusion_weights"].cpu().numpy().tolist()
    }

# Example:
# result = predict_images(
#     model,
#     ["/content/new_histology_200x.png"],
#     [200]
# )
# print(result)


## 20. Optional Gradio web demo

The web interface below is intentionally optional. It loads one image, performs the complete preprocessing/graph/prediction path and displays the benign/malignant probability. The source workflow describes deployment on constrained hardware and an ONNX/Raspberry Pi path; this notebook keeps the web demo separate from the core inference model. fileciteturn0file0L697-L703


In [ ]:
# Cell 30 — Optional Gradio demo

!pip -q install gradio

import gradio as gr

def gradio_predict(img):
    if img is None:
        return "Please upload a histopathology image.", None

    temp=os.path.join(CONFIG["OUTPUT_ROOT"],"gradio_input.png")
    Image.fromarray(np.asarray(img).astype(np.uint8)).save(temp)

    result=predict_images(model,[temp],[200])
    text_out=(
        f"Prediction: {result['prediction']}\n"
        f"Benign probability: {result['benign_probability']:.4f}\n"
        f"Malignant probability: {result['malignant_probability']:.4f}"
    )
    return text_out, img

demo=gr.Interface(
    fn=gradio_predict,
    inputs=gr.Image(type="numpy",label="Histopathology Image"),
    outputs=[
        gr.Textbox(label="MDG-LiteNet Result"),
        gr.Image(label="Input")
    ],
    title="MDG-LiteNet — Explainable Breast Tumor Classification",
    description="Macenko → Otsu → MORPH-G → DeformGraph → MAG-Fuse → sparse classifier"
)

# In Colab, run:
# demo.launch(share=True)
print("Gradio demo created. Uncomment demo.launch(share=True) to start it.")


## 21. Optional ONNX export

For ONNX deployment, the dynamic graph operations in this research implementation should first be wrapped into a fixed-shape inference graph. The following cell exports the compact classifier head as a deployment building block; the complete variable-node graph path is kept in PyTorch.


In [ ]:
# Cell 31 — Export compact head example

class CompactHeadWrapper(nn.Module):
    def __init__(self, head):
        super().__init__()
        self.head=head
    def forward(self,x):
        logits,_,_=self.head(x)
        return F.softmax(logits,dim=-1)

try:
    wrapper=CompactHeadWrapper(model.head).cpu().eval()
    dummy=torch.randn(1,CONFIG["WIDTH"])
    onnx_path=os.path.join(CONFIG["OUTPUT_ROOT"],"mdg_litenet_compact_head.onnx")
    torch.onnx.export(
        wrapper,dummy,onnx_path,
        input_names=["embedding"],output_names=["probability"],
        opset_version=17
    )
    print("ONNX head saved:",onnx_path)
except Exception as e:
    print("ONNX export skipped:",e)


## 22. Final artifact summary

Expected outputs under `CONFIG["OUTPUT_ROOT"]` include:
- `mdg_litenet_model.pt`
- `mdg_litenet_int8_state.pt`
- `config.json`
- `training_history.csv`
- `patient_predictions.csv`
- `magnification_subset_results.csv`
- graph cache
- XGraph-Histo decision maps and node attribution CSVs
- optional compact ONNX classifier head

This notebook is a runnable engineering implementation of the methodology described in the supplied workflow. The supplied manuscript itself specifies that the reported parameter/latency counts are analytical and should not replace measured latency; likewise, this notebook computes its own metrics from the actual data/run rather than copying manuscript values. fileciteturn0file0L634-L648
